In [49]:
import numpy as np
import pandas as pd

training_data = pd.read_csv('data/train.csv').to_numpy()
test_data = pd.read_csv('data/test.csv').to_numpy()

In [50]:
print(len(training_data))

X_train = training_data[:, 1:] / 255
Y_train = training_data[:, 0]
Y_train = np.eye(10)[Y_train]

m = 20000

X_sample = X_train[0:m].reshape(784,-1)
Y_sample = Y_train[0:m].reshape(10,-1)

print(X_sample.shape)
print(Y_sample.shape)

42000
(784, 20000)
(10, 20000)


In [51]:
def initialise_weights():

    W1 = np.random.rand(128,784) * np.sqrt(1 / 784) - 0.5
    b1 = np.random.rand(1,128).reshape(-1,1) - 0.5

    W2 = np.random.rand(64,128) * np.sqrt(1 / 784) - 0.5
    b2 = np.random.rand(1,64).reshape(-1,1) - 0.5

    W3 = np.random.rand(10,64) * np.sqrt(1 / 784) - 0.5
    b3 = np.random.rand(1, 10).reshape(-1,1) - 0.5

    return W1, b1, W2, b2, W3, b3

In [52]:
# Activation functions
def ReLU(Z):
    return np.tanh(Z)

def Softmax(Z):
    Z_exp = np.exp(Z)
    A = Z_exp / Z_exp.sum()
    return A

In [53]:
def forward_prop(W1, b1, W2, b2, W3, b3):
    # layer 1
    Z1 = W1 @ X_sample + b1
    # print("W1:        ",W1.shape)
    # print("X_sample:  ",X_sample.shape)
    # print("b1:        ",b1.shape)
    # print("Z1:        ",Z1.shape)
    A1 = ReLU(Z1)
    # print("A1:        ",A1.shape)
    # print("\n")

    # layer 2
    Z2 = W2 @ A1 + b2
    # print("W2:        ",W2.shape)
    # print("A2:        ",A1.shape)
    # print("b2:        ",b2.shape)
    # print("Z2:        ",Z2.shape)
    A2 = ReLU(Z2)
    # print("A2:        ",A2.shape)
    # print("\n")

    # layer 3
    Z3 = W3 @ A2 + b3
    # print("W3:        ",W3.shape)
    # print("A2:        ",A2.shape)
    # print("b3:        ",b3.shape)
    # print("Z3:        ",Z3.shape)
    A3 = Softmax(Z3)
    # print("A3:        ",A3.shape)
    # print("\n")

    return Z1, A1, Z2, A2, Z3, A3

In [54]:
def back_prop(Y_sample, X_sample, W1, W2, W3, A1, A2, A3, m):
    # calculate loss
    L = -Y_sample * np.log(A3).sum()

    # use loss to calculate the derivative of A3
    delta3 = dZ3 = A3 - Y_sample
    dW3 = (1/m) * delta3 @ A2.T
    db3 = np.sum(delta3, axis=1, keepdims=True)

    dA2 = np.dot(W3.T, dZ3)
    delta2 = dZ2 = np.maximum(0, dA2)
    dW2 = (1/m) * delta2 @ A1.T
    db2 = np.sum(delta2, axis=1, keepdims=True)
    
    dA1 = np.dot(W2.T, dZ2)
    delta1 = dZ1 = np.maximum(0, dA1)
    dW1 = (1/m) * delta1 @ X_sample.T
    db1 = np.sum(delta1, axis=1, keepdims=True)

    return dW3, db3, dW2, db2, dW1, db1, L

In [55]:
def update(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2, dW3, db3, alpha):
    W1 -= dW1 * alpha
    b1 -= db1 * alpha
    W2 -= dW2 * alpha
    b2 -= db2 * alpha
    W3 -= dW3 * alpha
    b3 -= db3 * alpha

    return W1, b1, W2, b2, W3, b3

In [59]:
def gradient_descent(X_sample, Y_sample, epoches, m):
    for epoch in range(epoches):
        print(epoch)
        W1, b1, W2, b2, W3, b3 = initialise_weights()
        Z1, A1, Z2, A2, Z3, A3 = forward_prop(W1, b1, W2, b2, W3, b3)
        dW3, db3, dW2, db2, dW1, db1, L = back_prop(Y_sample, X_sample, W1, W2, W3, A1, A2, A3, m)
        W1, b1, W2, b2, W3, b3 = update(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2, dW3, db3, 0.01)
        if epoch % 10 == 0:
            print(f"Epoch {epoch} completed.")
            print(f"Loss: {L}")

In [60]:
gradient_descent(X_sample, Y_sample, 100, m)

0
Epoch 0 completed.
Loss: [[      0.         2450998.07277145       0.         ...       0.
        0.               0.        ]
 [      0.               0.               0.         ...       0.
        0.               0.        ]
 [      0.               0.               0.         ...       0.
        0.               0.        ]
 ...
 [      0.               0.               0.         ...       0.
        0.               0.        ]
 [      0.         2450998.07277145       0.         ...       0.
        0.               0.        ]
 [      0.               0.               0.         ... 2450998.07277145
        0.               0.        ]]
1
2
3
4
5
6
7
8
9
10
Epoch 10 completed.
Loss: [[      0.         2444197.18049696       0.         ...       0.
        0.               0.        ]
 [      0.               0.               0.         ...       0.
        0.               0.        ]
 [      0.               0.               0.         ...       0.
        0.            